<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Document-ingestion" data-toc-modified-id="Document-ingestion-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Document ingestion</a></span><ul class="toc-item"><li><span><a href="#Test-the-Connection" data-toc-modified-id="Test-the-Connection-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Test the Connection</a></span></li><li><span><a href="#Test-the-Search" data-toc-modified-id="Test-the-Search-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Test the Search</a></span></li></ul></li><li><span><a href="#document_chunker" data-toc-modified-id="document_chunker-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>document_chunker</a></span></li><li><span><a href="#vector_store.py" data-toc-modified-id="vector_store.py-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>vector_store.py</a></span></li></ul></div>

In [1]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MCP Simple Example")

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    return a + b

# Show registered tools
print(mcp._tool_manager.list_tools())

[Tool(fn=<function add_numbers at 0x000001CB28737600>, name='add_numbers', title=None, description='', parameters={'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'add_numbersArguments', 'type': 'object'}, fn_metadata=FuncMetadata(arg_model=<class 'mcp.server.fastmcp.utilities.func_metadata.add_numbersArguments'>, output_schema={'properties': {'result': {'title': 'Result', 'type': 'integer'}}, 'required': ['result'], 'title': 'add_numbersOutput', 'type': 'object'}, output_model=<class 'mcp.server.fastmcp.utilities.func_metadata.add_numbersOutput'>, wrap_output=True), is_async=False, context_kwarg=None, annotations=None, icons=None, meta=None)]


In [2]:
# A way to get a tool from langchain
from langchain_community.agent_toolkits.load_tools import load_tools
# Core LLM interface (OpenAI chat models)
from langchain_openai import ChatOpenAI
from rich.markdown import Markdown
import os
from dotenv import load_dotenv
load_dotenv(override=True)
    
# Read API key from environment variable
os.environ["SERPAPI_API_KEY"] = os.getenv("SERP_API_KEY")  # https://serpapi.com for a free token!
# LLM Model
os.environ['OPENAI_API_KEY'] = os.environ.get("OPENAI_API_KEY")
model = ChatOpenAI(model='gpt-4o-mini')

tools = load_tools(["serpapi"])

In [3]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

print("Imports successful")

Imports successful


## Document ingestion

recursively scan your folders and extract content from PDF, CSV, Python, DOCX, Excel, notes, etc.

`document_search_mcp.py` is an **MCP server for searching files recursively within a directory**.
It searches both **file names and file contents** across PDF, DOCX, XLSX, CSV, Python, TXT, Markdown, JSON, and Jupyter files.

For PDFs, it extracts text using **PyMuPDF**; for Excel/DOCX it extracts their structured content.
It returns the **matching file, file type, match type, and surrounding context**.
This is currently the **document retrieval layer**; the next step is to add **chunking + embeddings + Chroma** to turn it into a true RAG system.


Here is the step-by-step test procedure for `document_search_mcp.py`.

* Open Anaconda Prompt and activate the environment where MCP is installed: `conda activate agentic_workflow`
  * Then run: `python "/.document_search_mcp.py"`
* Check that the MCP endpoint is available: `http://127.0.0.1:8000/mcp`
  * **Keep this terminal running.**

### Test the Connection

In [4]:
mcp_ctx = streamablehttp_client(
    "http://127.0.0.1:8000/mcp"
)

read_stream, write_stream, _ = await mcp_ctx.__aenter__()

session_ctx = ClientSession(
    read_stream,
    write_stream
)

session = await session_ctx.__aenter__()

await session.initialize()

print("MCP connection successful!")

[08/23/26 08:40:45] INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"         ]8;id=858270;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=51088;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     Received session ID: dbf1fe47ee284ceca70bfc3dbeea06e9           ]8;id=78286;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=672073;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=46016;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=879738;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py#193\193]8;;\

MCP connection successful!


In [5]:
tools_response = await session.list_tools()

print("===== MCP TOOLS =====")

for tool in tools_response.tools:
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description}")
    print()

                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"   ]8;id=970296;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=864271;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"          ]8;id=531916;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=67139;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"         ]8;id=8751;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=739316;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

===== MCP TOOLS =====
Tool: search_documents
Description: 
    Recursively search filenames and file contents.

    Searches supported documents including:

    PDF
    DOCX
    XLSX
    CSV
    TXT
    Markdown
    Python
    JSON
    XML
    HTML
    Jupyter notebooks
    and other text-based files.

    Returns JSON containing:

    - folders_searched
    - files_searched
    - number_of_results
    - matching documents
    - surrounding text context
    



### Test the Search 

In [6]:
import json


async def search_test():

    search_results = None

    try:

        async with streamablehttp_client(
            "http://127.0.0.1:8000/mcp"
        ) as (read_stream, write_stream, _):

            async with ClientSession(
                read_stream,
                write_stream
            ) as session:

                await session.initialize()

                result = await session.call_tool(
                    "search_documents",
                    {
                        "directory": r"E:\Buying House",
                        "query": "Real Property Report",
                        "max_results": 100
                    }
                )

                # Get raw MCP response
                raw_result = result.content[0].text

                # Convert JSON -> Python dictionary
                search_results = json.loads(
                    raw_result
                )

                print(
                    f"Found "
                    f"{search_results['number_of_results']} "
                    f"results"
                )

    except ExceptionGroup as e:

        # The MCP request completed, but
        # AnyIO raises an exception during cleanup.
        if search_results is not None:

            print(
                "MCP search completed. "
                "Ignoring Streamable HTTP cleanup error."
            )

        else:

            raise

    return search_results

In [7]:
search_results = await search_test()

                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"         ]8;id=599071;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=184335;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     Received session ID: 94f1ae387bfd4b9491131828619b217c           ]8;id=815980;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=251406;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=518941;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=175767;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py#193\193]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 202 Accepted"   ]8;id=633925;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=610332;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     HTTP Request: GET http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"          ]8;id=814411;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=816797;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"         ]8;id=852609;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=297531;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"         ]8;id=326500;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=558851;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     HTTP Request: DELETE http://127.0.0.1:8000/mcp "HTTP/1.1 200 OK"       ]8;id=384299;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=339472;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\httpx\_client.py#1786\1786]8;;\

                    INFO     GET stream disconnected, reconnecting in 1000ms...              ]8;id=219654;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py\streamable_http.py]8;;\:]8;id=518522;file://D:\Learning\MyWebsite\FinalGithub\ToPublihsed\projects\Agentic_Workflow\agentic_workflow\Lib\site-packages\mcp\client\streamable_http.py#298\298]8;;\

MCP search completed. Ignoring Streamable HTTP cleanup error.


## document_chunker

In [8]:
def chunk_text(
    text,
    chunk_size=1000,
    overlap=150
):

    chunks = []

    start = 0

    while start < len(text):

        end = min(
            start + chunk_size,
            len(text)
        )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks


def chunk_search_results(
    search_results,
    chunk_size=1000,
    overlap=150
):

    chunks = []

    for result in search_results["results"]:

        text = result["context"]

        text_chunks = chunk_text(
            text,
            chunk_size,
            overlap
        )

        for i, chunk in enumerate(
            text_chunks,
            start=1
        ):

            chunks.append({

                "text": chunk,

                "metadata": {

                    "source": result["file"],

                    "filename":
                        result["file_name"],

                    "file_type":
                        result["file_type"],

                    "match_type":
                        result["match_type"],

                    "chunk": i
                }
            })

    return chunks

In [9]:
#from document_chunker import chunk_search_results

chunks = chunk_search_results(
    search_results
)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:10]):

    print("=" * 80)

    print("CHUNK:", i + 1)

    print("\nSOURCE:")
    print(chunk["metadata"]["source"])

    print("\nTEXT:")

KeyError: 'results'

## vector_store.py

In [ ]:
import chromadb

from openai import OpenAI


# =========================================================
# OPENAI
# =========================================================

client_openai = OpenAI()


EMBEDDING_MODEL = "text-embedding-3-small"


# =========================================================
# CHROMA
# =========================================================

client_chroma = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client_chroma.get_or_create_collection(
    name="documents"
)


# =========================================================
# CREATE OPENAI EMBEDDINGS
# =========================================================

def create_embeddings(texts):

    response = client_openai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )

    return [
        item.embedding
        for item in response.data
    ]


# =========================================================
# ADD CHUNKS
# =========================================================

def add_chunks(chunks):

    if not chunks:
        return

    texts = []
    metadatas = []
    ids = []

    for i, chunk in enumerate(chunks):

        texts.append(
            chunk["text"]
        )

        metadatas.append(
            chunk["metadata"]
        )

        ids.append(
            f"chunk_{i}"
        )

    # OpenAI embeddings
    embeddings = create_embeddings(
        texts
    )

    collection.upsert(
        ids=ids,
        documents=texts,
        metadatas=metadatas,
        embeddings=embeddings
    )

    print(
        f"Added {len(chunks)} chunks to Chroma."
    )


add_chunks(chunks)
# =========================================================
# SEMANTIC SEARCH
# =========================================================

def search_chunks(
    query,
    n_results=5
):

    # Create embedding for user question
    query_embedding = create_embeddings(
        [query]
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding
        ],
        n_results=n_results
    )

    return results

In [ ]:
#pip install chromadb sentence-transformers

In [ ]:
results = search_chunks(
    "show me file name with utility bill",
    n_results=5
)

In [ ]:
print("Number of documents:", collection.count())

In [ ]:
for document, metadata in zip(
    results["documents"][0],
    results["metadatas"][0]
):

    print("=" * 80)

    print("SOURCE:")
    print(metadata["source"])

    print("\nTEXT:")
    print(document)

In [ ]:
from openai import OpenAI

client_openai = OpenAI()

CHAT_MODEL = "gpt-5-mini"


def rag_answer(
    question,
    n_results=5
):

    # -----------------------------------------
    # 1. Retrieve relevant chunks
    # -----------------------------------------

    results = search_chunks(
        question,
        n_results=n_results
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    if not documents:

        return "I could not find relevant information."


    # -----------------------------------------
    # 2. Build context
    # -----------------------------------------

    context_parts = []

    for i, (document, metadata) in enumerate(
        zip(documents, metadatas),
        start=1
    ):

        context_parts.append(
            f"""
SOURCE {i}:
{metadata["source"]}

CONTENT:
{document}
"""
        )

    context = "\n\n".join(
        context_parts
    )


    # -----------------------------------------
    # 3. Ask OpenAI
    # -----------------------------------------

    prompt = f"""
Answer the user's question using ONLY
the information provided in the context.

If the answer cannot be found in the context,
say that the information was not found.

USER QUESTION:
{question}

CONTEXT:
{context}
"""

    response = client_openai.responses.create(
        model=CHAT_MODEL,
        input=prompt
    )


    # -----------------------------------------
    # 4. Return answer
    # -----------------------------------------

    return response.output_text

In [ ]:
answer = rag_answer(
    "show me all pdf files that related to TAX"
)

print(answer)